In [1]:
!pip install requests feedparser newspaper4k pandas langdetect tqdm

     ---------------------------------------- 0.0/981.5 kB ? eta -:--:--
     ---------- ----------------------------- 262.1/981.5 kB ? eta -:--:--
     -------------------- ----------------- 524.3/981.5 kB 1.5 MB/s eta 0:00:01
     ---------------------------------------- 981.5/981.5 kB 1.5 MB/s  0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993364 sha256=92857c53bcc8cc219271fd0af3b25dfeff8b63f83283f5fa41a35d2e70622875
  Stored in directory: c:\users\lenovo\appdata\local\pip\cache\wheels\95\03\7d\59ea870c70ce4e5a370638b5462a7711ab78fba2f655d05106
Successfully built langdetect



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# =============================================================================
# SCRAPING BERITA INDONESIA UNTUK TOPIC MODELLING
# Target: 1000 artikel dari berbagai media Indonesia via RSS Feed
# Output: dataset_berita.csv (kolom: isi)
# =============================================================================

# --- INSTALASI LIBRARY ---
# Jalankan perintah ini di terminal / Jupyter cell pertama:
#
# pip install requests feedparser newspaper4k pandas langdetect tqdm
#
# Catatan:
# - newspaper4k = versi modern dari newspaper3k, lebih stabil untuk Python 3.10+
# - langdetect  = untuk filter artikel non-Indonesia
# - tqdm        = progress bar (opsional, sudah ada counter manual juga)

# =============================================================================
# IMPORT LIBRARY
# =============================================================================
import requests
import feedparser
import pandas as pd
import time
import re
import hashlib
import sys
from datetime import datetime

# newspaper4k untuk ekstrak full text dari URL
try:
    from newspaper import Article
    NEWSPAPER_OK = True
except ImportError:
    print("[WARN] newspaper4k belum terinstall. Jalankan: pip install newspaper4k")
    NEWSPAPER_OK = False

# langdetect untuk deteksi bahasa
try:
    from langdetect import detect, LangDetectException
    LANGDETECT_OK = True
except ImportError:
    print("[WARN] langdetect belum terinstall. Filter bahasa dinonaktifkan.")
    LANGDETECT_OK = False


# =============================================================================
# KONFIGURASI UTAMA
# =============================================================================
TARGET_ARTIKEL   = 1000       # Jumlah artikel yang ingin dikumpulkan
MIN_KARAKTER     = 200        # Minimal panjang teks artikel (karakter)
JEDA_ANTAR_URL   = 1.5        # Detik jeda antar request (rate limiting)
JEDA_ANTAR_FEED  = 3.0        # Detik jeda saat pindah ke feed berikutnya
TIMEOUT_REQUEST  = 15         # Timeout HTTP request (detik)
OUTPUT_FILE      = "dataset_berita.csv"

# Header HTTP agar tidak mudah diblok sebagai bot
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "id-ID,id;q=0.9,en-US;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}


# =============================================================================
# DAFTAR RSS FEED
# Prioritas berdasarkan stabilitas (paling stabil di atas)
# Format: ("Nama Media", "URL RSS Feed")
# =============================================================================
RSS_FEEDS = [
    # --- ANTARA: Paling stabil, jarang block, cocok jadi sumber utama ---
    ("Antara - Nasional",    "https://www.antaranews.com/rss/terkini.xml"),
    ("Antara - Ekonomi",     "https://www.antaranews.com/rss/ekonomi.xml"),
    ("Antara - Politik",     "https://www.antaranews.com/rss/politik.xml"),
    ("Antara - Hukum",       "https://www.antaranews.com/rss/hukum.xml"),
    ("Antara - Pendidikan",  "https://www.antaranews.com/rss/pendidikan.xml"),

    # --- REPUBLIKA: Cukup stabil ---
    ("Republika - Nasional", "https://www.republika.co.id/rss/nasional"),
    ("Republika - Ekonomi",  "https://www.republika.co.id/rss/ekonomi"),
    ("Republika - Pendidikan","https://www.republika.co.id/rss/pendidikan"),

    # --- TEMPO: Alternatif CNN Indonesia, lebih stabil ---
    ("Tempo - Nasional",     "https://www.tempo.co/rss/nasional"),
    ("Tempo - Ekonomi",      "https://www.tempo.co/rss/bisnis"),
    ("Tempo - Hukum",        "https://www.tempo.co/rss/hukum"),

    # --- KOMPAS: Stabil tapi kadang throttle, jadikan buffer ---
    ("Kompas - News",        "https://rss.kompas.com/index.php/rss/json2rss/https://www.kompas.com/tag/berita-terkini"),
    ("Kompas - Regional",    "https://regional.kompas.com/rss/20170815063435"),
    ("Kompas - Ekonomi",     "https://money.kompas.com/rss/20170815063435"),

    # --- TRIBUNNEWS: Volume besar, cocok sebagai backup ---
    ("Tribunnews - Nasional","https://www.tribunnews.com/rss/nasional"),
    ("Tribunnews - Ekonomi", "https://www.tribunnews.com/rss/bisnis-dan-keuangan"),
    ("Tribunnews - Regional","https://www.tribunnews.com/rss/regional"),

    # --- DETIK: Kadang block, taruh di bawah sebagai backup ---
    ("Detik - News",         "https://rss.detik.com/index.php/detikcom"),
    ("Detik - Finance",      "https://rss.detik.com/index.php/finance"),
    ("Detik - Hot",          "https://rss.detik.com/index.php/hot"),

    # --- KUMPARAN: Alternatif tambahan ---
    ("Kumparan - News",      "https://kumparan.com/kumparannews/rss"),
    ("Kumparan - Bisnis",    "https://kumparan.com/kumparan-bisnis/rss"),
]

# Catatan RSS yang TIDAK dipakai dan alasannya:
# - CNN Indonesia (cnnindonesia.com): RSS feed tidak konsisten, sering 403/timeout
#   Alternatif: gunakan Tempo atau Antara sebagai pengganti
# - Okezone: Struktur RSS berubah-ubah, artikel sering terblok
# - Liputan6: Kadang redirect loop saat ekstrak full text


# =============================================================================
# FUNGSI UTILITAS
# =============================================================================

def bersihkan_teks(teks: str) -> str:
    """
    Membersihkan teks dari karakter tidak perlu.
    - Hapus whitespace berlebih
    - Hapus newline ganda
    - Strip leading/trailing space
    """
    if not teks:
        return ""
    # Ganti newline dan tab jadi spasi tunggal
    teks = re.sub(r"[\n\r\t]+", " ", teks)
    # Ganti spasi berlebih jadi satu spasi
    teks = re.sub(r" {2,}", " ", teks)
    return teks.strip()


def hash_teks(teks: str) -> str:
    """Membuat hash MD5 dari teks untuk deteksi duplikat."""
    return hashlib.md5(teks.encode("utf-8")).hexdigest()


def adalah_bahasa_indonesia(teks: str) -> bool:
    """
    Mendeteksi apakah teks berbahasa Indonesia.
    Return True jika Indonesia atau jika langdetect tidak tersedia.
    """
    if not LANGDETECT_OK:
        return True  # Skip filter jika library tidak ada
    try:
        bahasa = detect(teks[:500])  # Cukup cek 500 karakter pertama
        return bahasa == "id"
    except LangDetectException:
        return True  # Jika gagal deteksi, anggap valid


def ekstrak_artikel(url: str) -> str:
    """
    Mengekstrak isi teks artikel dari URL menggunakan newspaper4k.
    Return string teks atau "" jika gagal.
    """
    if not NEWSPAPER_OK:
        return ""
    try:
        artikel = Article(url, language="id")
        artikel.download()
        artikel.parse()
        return artikel.text
    except Exception:
        return ""


def ambil_url_dari_feed(url_feed: str) -> list:
    """
    Mengambil daftar URL artikel dari RSS feed.
    Return list of URL strings.
    """
    try:
        # feedparser otomatis handle parsing RSS/Atom
        feed = feedparser.parse(url_feed)
        urls = []
        for entry in feed.entries:
            link = getattr(entry, "link", None)
            if link:
                urls.append(link)
        return urls
    except Exception as e:
        print(f"  [ERROR] Gagal parse feed: {e}")
        return []


# =============================================================================
# FUNGSI UTAMA: SCRAPING
# =============================================================================

def jalankan_scraping():
    """
    Fungsi utama yang menjalankan seluruh proses scraping.
    Loop melalui setiap RSS feed sampai target artikel tercapai.
    """
    print("=" * 60)
    print("  SCRAPING BERITA INDONESIA - TOPIC MODELLING DATASET")
    print(f"  Target  : {TARGET_ARTIKEL} artikel")
    print(f"  Output  : {OUTPUT_FILE}")
    print(f"  Mulai   : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 60)

    # Kumpulkan data di sini
    data_artikel   = []   # List of string (isi artikel)
    hash_duplikat  = set() # Set hash untuk cek duplikat
    total_gagal    = 0    # Counter URL yang gagal
    total_dicoba   = 0    # Counter URL yang dicoba

    # Loop setiap RSS feed
    for nama_media, url_feed in RSS_FEEDS:

        # Cek apakah target sudah tercapai
        if len(data_artikel) >= TARGET_ARTIKEL:
            break

        print(f"\n[FEED] {nama_media}")
        print(f"  URL  : {url_feed}")

        # Ambil daftar URL dari feed
        daftar_url = ambil_url_dari_feed(url_feed)
        print(f"  Artikel ditemukan di feed: {len(daftar_url)}")

        if not daftar_url:
            print(f"  [SKIP] Feed kosong atau tidak bisa diakses.")
            time.sleep(JEDA_ANTAR_FEED)
            continue

        # Loop setiap URL artikel dalam feed
        for url_artikel in daftar_url:

            # Stop jika target tercapai
            if len(data_artikel) >= TARGET_ARTIKEL:
                break

            total_dicoba += 1
            nomor_sekarang = len(data_artikel) + 1

            # Tampilkan progress counter
            print(
                f"  [{nomor_sekarang:04d}/{TARGET_ARTIKEL}] "
                f"Mencoba: {url_artikel[:70]}...",
                end="", flush=True
            )

            # --- Ekstrak teks artikel ---
            teks = ekstrak_artikel(url_artikel)

            # Validasi 1: Teks tidak kosong
            if not teks:
                print(" -> [GAGAL] Teks kosong")
                total_gagal += 1
                time.sleep(JEDA_ANTAR_URL)
                continue

            # Bersihkan teks
            teks = bersihkan_teks(teks)

            # Validasi 2: Minimal panjang karakter
            if len(teks) < MIN_KARAKTER:
                print(f" -> [SKIP] Terlalu pendek ({len(teks)} kar)")
                total_gagal += 1
                time.sleep(JEDA_ANTAR_URL)
                continue

            # Validasi 3: Cek duplikat via hash
            h = hash_teks(teks)
            if h in hash_duplikat:
                print(" -> [SKIP] Duplikat")
                time.sleep(JEDA_ANTAR_URL)
                continue

            # Validasi 4: Cek bahasa Indonesia
            if not adalah_bahasa_indonesia(teks):
                print(" -> [SKIP] Bukan Bahasa Indonesia")
                total_gagal += 1
                time.sleep(JEDA_ANTAR_URL)
                continue

            # Semua validasi lolos — simpan artikel
            hash_duplikat.add(h)
            data_artikel.append(teks)
            print(f" -> [OK] {len(teks)} karakter (total: {len(data_artikel)})")

            # Jeda antar request (rate limiting)
            time.sleep(JEDA_ANTAR_URL)

        # Jeda lebih lama saat pindah ke feed berikutnya
        time.sleep(JEDA_ANTAR_FEED)

    # ==========================================================================
    # SIMPAN KE CSV
    # ==========================================================================
    print("\n" + "=" * 60)
    print("  MENYIMPAN HASIL...")
    print("=" * 60)

    if data_artikel:
        df = pd.DataFrame({"isi": data_artikel})
        df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
        # utf-8-sig menambahkan BOM agar Excel bisa baca karakter Indonesia dengan benar
        print(f"  File disimpan: {OUTPUT_FILE}")
    else:
        print("  [WARN] Tidak ada artikel yang berhasil dikumpulkan!")

    # ==========================================================================
    # RINGKASAN AKHIR
    # ==========================================================================
    print("\n" + "=" * 60)
    print("  RINGKASAN AKHIR")
    print("=" * 60)
    print(f"  Total URL dicoba          : {total_dicoba}")
    print(f"  Total artikel BERHASIL    : {len(data_artikel)}")
    print(f"  Total artikel GAGAL/SKIP  : {total_gagal}")
    print(f"  Target tercapai           : {'YA ✓' if len(data_artikel) >= TARGET_ARTIKEL else 'BELUM ✗'}")
    if data_artikel:
        print(f"  Rata-rata panjang teks    : {sum(len(t) for t in data_artikel) // len(data_artikel)} karakter")
    print(f"  Selesai                   : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 60)

    if len(data_artikel) < TARGET_ARTIKEL:
        kekurangan = TARGET_ARTIKEL - len(data_artikel)
        print(f"\n[INFO] Kurang {kekurangan} artikel dari target.")
        print("  Saran:")
        print("  1. Tambah RSS feed lain di daftar RSS_FEEDS")
        print("  2. Kurangi MIN_KARAKTER (misal jadi 100)")
        print("  3. Jalankan ulang script — feed yang sama bisa punya artikel berbeda")

    return data_artikel


# =============================================================================
# ENTRY POINT
# =============================================================================
if __name__ == "__main__":
    # Cek library penting
    if not NEWSPAPER_OK:
        print("[ERROR] newspaper4k diperlukan. Install dulu:")
        print("  pip install newspaper4k")
        sys.exit(1)

    jalankan_scraping()

  SCRAPING BERITA INDONESIA - TOPIC MODELLING DATASET
  Target  : 1000 artikel
  Output  : dataset_berita.csv
  Mulai   : 2026-04-24 23:14:36

[FEED] Antara - Nasional
  URL  : https://www.antaranews.com/rss/terkini.xml
  Artikel ditemukan di feed: 50
  [0001/1000] Mencoba: https://www.antaranews.com/berita/5541675/alwi-farhan-dan-raymond-joaq... -> [OK] 2242 karakter (total: 1)
  [0002/1000] Mencoba: https://www.antaranews.com/berita/5541673/esdm-tawarkan-116-blok-migas... -> [OK] 3404 karakter (total: 2)
  [0003/1000] Mencoba: https://www.antaranews.com/berita/5541669/bpom-perkuat-regulasi-kolabo... -> [OK] 3001 karakter (total: 3)
  [0004/1000] Mencoba: https://www.antaranews.com/video/5541625/revitalisasi-pasar-bogor-disi... -> [OK] 348 karakter (total: 4)
  [0005/1000] Mencoba: https://www.antaranews.com/berita/5541667/tekanan-global-meningkat-bi-... -> [OK] 3384 karakter (total: 5)
  [0006/1000] Mencoba: https://www.antaranews.com/berita/5541663/purbaya-usulkan-insentif-mot... ->

In [3]:
# =============================================================================
# SCRAPING BERITA INDONESIA UNTUK TOPIC MODELLING
# Target: 1000 artikel dari berbagai media Indonesia via RSS Feed
# Output: dataset_berita.csv (kolom: isi)
# =============================================================================

# --- INSTALASI LIBRARY ---
# Jalankan perintah ini di terminal / Jupyter cell pertama:
#
# pip install requests feedparser newspaper4k pandas langdetect tqdm
#
# Catatan:
# - newspaper4k = versi modern dari newspaper3k, lebih stabil untuk Python 3.10+
# - langdetect  = untuk filter artikel non-Indonesia
# - tqdm        = progress bar (opsional, sudah ada counter manual juga)

# =============================================================================
# IMPORT LIBRARY
# =============================================================================
import requests
import feedparser
import pandas as pd
import time
import re
import hashlib
import sys
from datetime import datetime

# newspaper4k untuk ekstrak full text dari URL
try:
    from newspaper import Article
    NEWSPAPER_OK = True
except ImportError:
    print("[WARN] newspaper4k belum terinstall. Jalankan: pip install newspaper4k")
    NEWSPAPER_OK = False

# langdetect untuk deteksi bahasa
try:
    from langdetect import detect, LangDetectException
    LANGDETECT_OK = True
except ImportError:
    print("[WARN] langdetect belum terinstall. Filter bahasa dinonaktifkan.")
    LANGDETECT_OK = False


# =============================================================================
# KONFIGURASI UTAMA
# =============================================================================
TARGET_ARTIKEL   = 1000       # Jumlah artikel yang ingin dikumpulkan
MIN_KARAKTER     = 200        # Minimal panjang teks artikel (karakter)
JEDA_ANTAR_URL   = 1.5        # Detik jeda antar request (rate limiting)
JEDA_ANTAR_FEED  = 3.0        # Detik jeda saat pindah ke feed berikutnya
TIMEOUT_REQUEST  = 15         # Timeout HTTP request (detik)
OUTPUT_FILE      = "dataset_berita_2.csv"

# Header HTTP agar tidak mudah diblok sebagai bot
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "id-ID,id;q=0.9,en-US;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}


# =============================================================================
# DAFTAR RSS FEED — 80+ FEED DARI 15+ MEDIA INDONESIA
# Diurutkan dari yang paling stabil ke yang paling sering bermasalah.
# Format: ("Nama Media - Kategori", "URL RSS Feed")
# =============================================================================
RSS_FEEDS = [

    # =========================================================================
    # [1] ANTARA NEWS — Kantor berita negara, PALING STABIL, jarang block
    #     Volume: ~30-50 artikel/hari per feed
    # =========================================================================
    ("Antara - Terkini",         "https://www.antaranews.com/rss/terkini.xml"),
    ("Antara - Ekonomi",         "https://www.antaranews.com/rss/ekonomi.xml"),
    ("Antara - Politik",         "https://www.antaranews.com/rss/politik.xml"),
    ("Antara - Hukum",           "https://www.antaranews.com/rss/hukum.xml"),
    ("Antara - Pendidikan",      "https://www.antaranews.com/rss/pendidikan.xml"),
    ("Antara - Olahraga",        "https://www.antaranews.com/rss/olahraga.xml"),
    ("Antara - Hiburan",         "https://www.antaranews.com/rss/hiburan.xml"),
    ("Antara - Teknologi",       "https://www.antaranews.com/rss/teknologi.xml"),
    ("Antara - Humaniora",       "https://www.antaranews.com/rss/humaniora.xml"),
    ("Antara - Internasional",   "https://www.antaranews.com/rss/internasional.xml"),
    ("Antara - Nusantara",       "https://www.antaranews.com/rss/nusantara.xml"),
    ("Antara - Lifestyle",       "https://www.antaranews.com/rss/lifestyle.xml"),
    ("Antara - Kesehatan",       "https://www.antaranews.com/rss/kesehatan.xml"),
    ("Antara - Lingkungan",      "https://www.antaranews.com/rss/lingkungan.xml"),
    ("Antara - Pariwisata",      "https://www.antaranews.com/rss/pariwisata.xml"),
    ("Antara - Pertanian",       "https://www.antaranews.com/rss/pertanian.xml"),
    ("Antara - Otomotif",        "https://www.antaranews.com/rss/otomotif.xml"),
    ("Antara - Sains",           "https://www.antaranews.com/rss/sains.xml"),

    # =========================================================================
    # [2] REPUBLIKA — Media Islam terbesar, stabil, konten beragam
    #     Volume: ~20-40 artikel/hari per feed
    # =========================================================================
    ("Republika - Nasional",     "https://www.republika.co.id/rss/nasional"),
    ("Republika - Ekonomi",      "https://www.republika.co.id/rss/ekonomi"),
    ("Republika - Internasional","https://www.republika.co.id/rss/internasional"),
    ("Republika - Olahraga",     "https://www.republika.co.id/rss/olahraga"),
    ("Republika - Teknologi",    "https://www.republika.co.id/rss/teknologi"),
    ("Republika - Pendidikan",   "https://www.republika.co.id/rss/pendidikan"),
    ("Republika - Gaya Hidup",   "https://www.republika.co.id/rss/gaya-hidup"),
    ("Republika - Islam",        "https://www.republika.co.id/rss/islam"),
    ("Republika - Hiburan",      "https://www.republika.co.id/rss/hiburan"),
    ("Republika - Khazanah",     "https://www.republika.co.id/rss/khazanah"),

    # =========================================================================
    # [3] TEMPO — Media independen terpercaya, konten investigatif
    #     Volume: ~15-30 artikel/hari per feed
    # =========================================================================
    ("Tempo - Nasional",         "https://www.tempo.co/rss/nasional"),
    ("Tempo - Bisnis",           "https://www.tempo.co/rss/bisnis"),
    ("Tempo - Hukum",            "https://www.tempo.co/rss/hukum"),
    ("Tempo - Politik",          "https://www.tempo.co/rss/politik"),
    ("Tempo - Internasional",    "https://www.tempo.co/rss/dunia"),
    ("Tempo - Teknologi",        "https://www.tempo.co/rss/tekno"),
    ("Tempo - Olahraga",         "https://www.tempo.co/rss/olahraga"),
    ("Tempo - Gaya Hidup",       "https://www.tempo.co/rss/gaya-hidup"),
    ("Tempo - Sains",            "https://www.tempo.co/rss/sains"),
    ("Tempo - Otomotif",         "https://www.tempo.co/rss/otomotif"),
    ("Tempo - Pendidikan",       "https://www.tempo.co/rss/pendidikan"),
    ("Tempo - Kesehatan",        "https://www.tempo.co/rss/kesehatan"),
    ("Tempo - Hiburan",          "https://www.tempo.co/rss/hiburan"),
    ("Tempo - Ekonomi Bisnis",   "https://bisnis.tempo.co/rss"),

    # =========================================================================
    # [4] TRIBUNNEWS — Volume SANGAT BESAR, konten regional kuat
    #     Volume: ~50-100 artikel/hari per feed
    # =========================================================================
    ("Tribun - Nasional",        "https://www.tribunnews.com/rss/nasional"),
    ("Tribun - Regional",        "https://www.tribunnews.com/rss/regional"),
    ("Tribun - Bisnis",          "https://www.tribunnews.com/rss/bisnis-dan-keuangan"),
    ("Tribun - Olahraga",        "https://www.tribunnews.com/rss/superskor"),
    ("Tribun - Sport",           "https://www.tribunnews.com/rss/sport"),
    ("Tribun - Techno",          "https://www.tribunnews.com/rss/techno"),
    ("Tribun - Lifestyle",       "https://www.tribunnews.com/rss/lifestyle"),
    ("Tribun - Seleb",           "https://www.tribunnews.com/rss/seleb"),
    ("Tribun - Travel",          "https://www.tribunnews.com/rss/travel"),
    ("Tribun - Kesehatan",       "https://www.tribunnews.com/rss/kesehatan"),
    ("Tribun - Pendidikan",      "https://www.tribunnews.com/rss/pendidikan"),
    ("Tribun - Otomotif",        "https://www.tribunnews.com/rss/otomotif"),
    ("Tribun - Hukum",           "https://www.tribunnews.com/rss/hukum"),
    ("Tribun - Internasional",   "https://www.tribunnews.com/rss/internasional"),
    # Sub-domain Tribun (kota-kota)
    ("Tribun Jatim",             "https://jatim.tribunnews.com/rss"),
    ("Tribun Jabar",             "https://jabar.tribunnews.com/rss"),
    ("Tribun Jakarta",           "https://jakarta.tribunnews.com/rss"),
    ("Tribun Sumsel",            "https://sumsel.tribunnews.com/rss"),
    ("Tribun Jateng",            "https://jateng.tribunnews.com/rss"),
    ("Tribun Bali",              "https://bali.tribunnews.com/rss"),
    ("Tribun Medan",             "https://medan.tribunnews.com/rss"),
    ("Tribun Manado",            "https://manado.tribunnews.com/rss"),
    ("Tribun Kaltim",            "https://kaltim.tribunnews.com/rss"),
    ("Tribun Timur",             "https://makassar.tribunnews.com/rss"),

    # =========================================================================
    # [5] KOMPAS — Media terpercaya, kualitas tinggi
    #     Catatan: Kadang throttle, tapi masih bisa
    # =========================================================================
    ("Kompas - Terkini",         "https://www.kompas.com/rss/20170519130927"),
    ("Kompas - Nasional",        "https://nasional.kompas.com/rss/20170815063435"),
    ("Kompas - Regional",        "https://regional.kompas.com/rss/20170815063435"),
    ("Kompas - Megapolitan",     "https://megapolitan.kompas.com/rss/20170815063435"),
    ("Kompas - Ekonomi",         "https://money.kompas.com/rss/20170815063435"),
    ("Kompas - Bola",            "https://bola.kompas.com/rss/20170815063435"),
    ("Kompas - Tekno",           "https://tekno.kompas.com/rss/20170815063435"),
    ("Kompas - Sains",           "https://sains.kompas.com/rss/20170815063435"),
    ("Kompas - Internasional",   "https://internasional.kompas.com/rss/20170815063435"),
    ("Kompas - Lifestyle",       "https://lifestyle.kompas.com/rss/20170815063435"),
    ("Kompas - Kesehatan",       "https://health.kompas.com/rss/20170815063435"),
    ("Kompas - Travel",          "https://travel.kompas.com/rss/20170815063435"),
    ("Kompas - Properti",        "https://properti.kompas.com/rss/20170815063435"),
    ("Kompas - Otomotif",        "https://otomotif.kompas.com/rss/20170815063435"),
    ("Kompas - Edukasi",         "https://edukasi.kompas.com/rss/20170815063435"),
    ("Kompas - Food",            "https://food.kompas.com/rss/20170815063435"),
    ("Kompas - Hype",            "https://hype.kompas.com/rss/20170815063435"),

    # =========================================================================
    # [6] KUMPARAN — Platform berita modern, konten beragam
    #     Volume: ~20-30 artikel/hari per feed
    # =========================================================================
    ("Kumparan - News",          "https://kumparan.com/kumparannews/rss"),
    ("Kumparan - Bisnis",        "https://kumparan.com/kumparan-bisnis/rss"),
    ("Kumparan - Sport",         "https://kumparan.com/kumparansport/rss"),
    ("Kumparan - Tekno",         "https://kumparan.com/kumparantech/rss"),
    ("Kumparan - Otomotif",      "https://kumparan.com/kumparanotomotif/rss"),
    ("Kumparan - Bola",          "https://kumparan.com/kumparanbola/rss"),
    ("Kumparan - Hits",          "https://kumparan.com/kumparanhits/rss"),
    ("Kumparan - Wanita",        "https://kumparan.com/kumparanwoman/rss"),
    ("Kumparan - Food",          "https://kumparan.com/kumparanfood/rss"),
    ("Kumparan - Moms",          "https://kumparan.com/kumparanmom/rss"),

    # =========================================================================
    # [7] CNBC INDONESIA — Fokus ekonomi & bisnis, kualitas tinggi
    # =========================================================================
    ("CNBC Indonesia - Market",  "https://www.cnbcindonesia.com/RSS"),
    ("CNBC Indonesia - Ekonomi", "https://www.cnbcindonesia.com/ekonomi/rss"),
    ("CNBC Indonesia - Bisnis",  "https://www.cnbcindonesia.com/entrepreneur/rss"),
    ("CNBC Indonesia - Invest",  "https://www.cnbcindonesia.com/investment/rss"),
    ("CNBC Indonesia - Tech",    "https://www.cnbcindonesia.com/tech/rss"),
    ("CNBC Indonesia - Dunia",   "https://www.cnbcindonesia.com/world/rss"),
    ("CNBC Indonesia - Lifestyle","https://www.cnbcindonesia.com/lifestyle/rss"),

    # =========================================================================
    # [8] DETIK — Berita cepat, volume besar
    #     Catatan: Kadang block bot, tapi masih bisa dengan User-Agent yang benar
    # =========================================================================
    ("Detik - News",             "https://rss.detik.com/index.php/detikcom"),
    ("Detik - Finance",          "https://rss.detik.com/index.php/finance"),
    ("Detik - Hot",              "https://rss.detik.com/index.php/hot"),
    ("Detik - Sport",            "https://rss.detik.com/index.php/sport"),
    ("Detik - Inet",             "https://rss.detik.com/index.php/inet"),
    ("Detik - Oto",              "https://rss.detik.com/index.php/oto"),
    ("Detik - Health",           "https://rss.detik.com/index.php/health"),
    ("Detik - Food",             "https://rss.detik.com/index.php/food"),
    ("Detik - Travel",           "https://rss.detik.com/index.php/travel"),
    ("Detik - Wolipop",          "https://rss.detik.com/index.php/wolipop"),
    ("Detik - X",                "https://rss.detik.com/index.php/detikx"),

    # =========================================================================
    # [9] OKEZONE — Berita umum, volume besar
    # =========================================================================
    ("Okezone - News",           "https://sindikasi.okezone.com/index.php/rss/1/OK"),
    ("Okezone - Economy",        "https://sindikasi.okezone.com/index.php/rss/2/OK"),
    ("Okezone - Lifestyle",      "https://sindikasi.okezone.com/index.php/rss/3/OK"),
    ("Okezone - Sports",         "https://sindikasi.okezone.com/index.php/rss/4/OK"),
    ("Okezone - Celebrity",      "https://sindikasi.okezone.com/index.php/rss/7/OK"),
    ("Okezone - Auto",           "https://sindikasi.okezone.com/index.php/rss/9/OK"),
    ("Okezone - Techno",         "https://sindikasi.okezone.com/index.php/rss/5/OK"),

    # =========================================================================
    # [10] LIPUTAN6 — Berita populer, konten luas
    # =========================================================================
    ("Liputan6 - News",          "https://www.liputan6.com/rss"),
    ("Liputan6 - Bisnis",        "https://www.liputan6.com/rss/bisnis"),
    ("Liputan6 - Bola",          "https://www.liputan6.com/rss/bola"),
    ("Liputan6 - Tekno",         "https://www.liputan6.com/rss/tekno"),
    ("Liputan6 - Lifestyle",     "https://www.liputan6.com/rss/lifestyle"),
    ("Liputan6 - Kesehatan",     "https://www.liputan6.com/rss/kesehatan"),
    ("Liputan6 - Otomotif",      "https://www.liputan6.com/rss/otomotif"),
    ("Liputan6 - Regional",      "https://www.liputan6.com/rss/regional"),
    ("Liputan6 - Global",        "https://www.liputan6.com/rss/global"),
    ("Liputan6 - Showbiz",       "https://www.liputan6.com/rss/showbiz"),

    # =========================================================================
    # [11] SUARA.COM — Media digital, konten beragam
    # =========================================================================
    ("Suara - News",             "https://www.suara.com/rss/news"),
    ("Suara - Bisnis",           "https://www.suara.com/rss/bisnis"),
    ("Suara - Bola",             "https://bola.suara.com/rss"),
    ("Suara - Tekno",            "https://www.suara.com/rss/tekno"),
    ("Suara - Lifestyle",        "https://www.suara.com/rss/lifestyle"),
    ("Suara - Health",           "https://www.suara.com/rss/health"),
    ("Suara - Entertainment",    "https://www.suara.com/rss/entertainment"),
    ("Suara - Otomotif",         "https://otomotif.suara.com/rss"),

    # =========================================================================
    # [12] MEDCOM — Media milik MNC Group, fokus berita nasional
    # =========================================================================
    ("Medcom - Nasional",        "https://www.medcom.id/rss/nasional"),
    ("Medcom - Ekonomi",         "https://www.medcom.id/rss/ekonomi"),
    ("Medcom - Internasional",   "https://www.medcom.id/rss/internasional"),
    ("Medcom - Olahraga",        "https://www.medcom.id/rss/olahraga"),
    ("Medcom - Teknologi",       "https://www.medcom.id/rss/teknologi"),

    # =========================================================================
    # [13] BISNIS.COM — Fokus ekonomi dan bisnis
    # =========================================================================
    ("Bisnis.com - Ekonomi",     "https://ekonomi.bisnis.com/rss"),
    ("Bisnis.com - Finansial",   "https://finansial.bisnis.com/rss"),
    ("Bisnis.com - Industri",    "https://industri.bisnis.com/rss"),
    ("Bisnis.com - Kabar24",     "https://kabar24.bisnis.com/rss"),
    ("Bisnis.com - Teknologi",   "https://teknologi.bisnis.com/rss"),
    ("Bisnis.com - Lifestyle",   "https://lifestyle.bisnis.com/rss"),
    ("Bisnis.com - Market",      "https://market.bisnis.com/rss"),

    # =========================================================================
    # [14] SINDONEWS — Media nasional dari MNC Group
    # =========================================================================
    ("Sindonews - Nasional",     "https://nasional.sindonews.com/rssfeeds/0/0"),
    ("Sindonews - Internasional","https://international.sindonews.com/rssfeeds/0/0"),
    ("Sindonews - Ekonomi",      "https://ekbis.sindonews.com/rssfeeds/0/0"),
    ("Sindonews - Tekno",        "https://tekno.sindonews.com/rssfeeds/0/0"),
    ("Sindonews - Sports",       "https://sports.sindonews.com/rssfeeds/0/0"),
    ("Sindonews - Otomotif",     "https://otomotif.sindonews.com/rssfeeds/0/0"),
    ("Sindonews - Islam",        "https://islam.sindonews.com/rssfeeds/0/0"),
    ("Sindonews - Selebritis",   "https://selebritis.sindonews.com/rssfeeds/0/0"),
    ("Sindonews - Metro",        "https://metro.sindonews.com/rssfeeds/0/0"),

    # =========================================================================
    # [15] MERDEKA.COM — Berita umum, volume tinggi
    # =========================================================================
    ("Merdeka - News",           "https://www.merdeka.com/feed/rss"),
    ("Merdeka - Bisnis",         "https://www.merdeka.com/uang/rss"),
    ("Merdeka - Teknologi",      "https://www.merdeka.com/teknologi/rss"),
    ("Merdeka - Olahraga",       "https://www.merdeka.com/olahraga/rss"),
    ("Merdeka - Gaya Hidup",     "https://www.merdeka.com/gaya/rss"),
    ("Merdeka - Sehat",          "https://www.merdeka.com/sehat/rss"),

    # =========================================================================
    # [16] VIVA — Berita dan entertainment
    # =========================================================================
    ("VIVA - Nasional",          "https://www.viva.co.id/rss/nasional"),
    ("VIVA - Bisnis",            "https://www.viva.co.id/rss/bisnis"),
    ("VIVA - Dunia",             "https://www.viva.co.id/rss/dunia"),
    ("VIVA - Bola",              "https://www.viva.co.id/rss/bola"),
    ("VIVA - Teknologi",         "https://www.viva.co.id/rss/digital"),
    ("VIVA - Otomotif",          "https://www.viva.co.id/rss/otomotif"),
    ("VIVA - Showbiz",           "https://www.viva.co.id/rss/showbiz"),
    ("VIVA - Gaya Hidup",        "https://www.viva.co.id/rss/gaya-hidup"),

]

# Catatan penting per media:
# ✅ Antara     : Paling stabil, tidak pernah block, prioritas utama
# ✅ Republika  : Stabil, konten variatif termasuk keislaman
# ✅ Tempo      : Kualitas tinggi, jarang block
# ✅ Tribun     : Volume terbesar, ada sub-domain kota
# ✅ CNBC Indo  : Bagus untuk topik ekonomi/bisnis
# ⚠️ Kompas    : Sesekali throttle, tapi umumnya masih bisa
# ⚠️ Detik     : Kadang block, tapi bisa lewat User-Agent yang benar
# ⚠️ Okezone   : Struktur RSS kadang berubah
# ⚠️ Liputan6  : Sesekali redirect loop
# ❌ CNN Indo  : Tidak dimasukkan — RSS sering 403/timeout


# =============================================================================
# FUNGSI UTILITAS
# =============================================================================

def bersihkan_teks(teks: str) -> str:
    """
    Membersihkan teks dari karakter tidak perlu.
    - Hapus whitespace berlebih
    - Hapus newline ganda
    - Strip leading/trailing space
    """
    if not teks:
        return ""
    # Ganti newline dan tab jadi spasi tunggal
    teks = re.sub(r"[\n\r\t]+", " ", teks)
    # Ganti spasi berlebih jadi satu spasi
    teks = re.sub(r" {2,}", " ", teks)
    return teks.strip()


def hash_teks(teks: str) -> str:
    """Membuat hash MD5 dari teks untuk deteksi duplikat."""
    return hashlib.md5(teks.encode("utf-8")).hexdigest()


def adalah_bahasa_indonesia(teks: str) -> bool:
    """
    Mendeteksi apakah teks berbahasa Indonesia.
    Return True jika Indonesia atau jika langdetect tidak tersedia.
    """
    if not LANGDETECT_OK:
        return True  # Skip filter jika library tidak ada
    try:
        bahasa = detect(teks[:500])  # Cukup cek 500 karakter pertama
        return bahasa == "id"
    except LangDetectException:
        return True  # Jika gagal deteksi, anggap valid


def ekstrak_artikel(url: str) -> str:
    """
    Mengekstrak isi teks artikel dari URL menggunakan newspaper4k.
    Return string teks atau "" jika gagal.
    """
    if not NEWSPAPER_OK:
        return ""
    try:
        artikel = Article(url, language="id")
        artikel.download()
        artikel.parse()
        return artikel.text
    except Exception:
        return ""


def ambil_url_dari_feed(url_feed: str) -> list:
    """
    Mengambil daftar URL artikel dari RSS feed.
    Return list of URL strings.
    """
    try:
        # feedparser otomatis handle parsing RSS/Atom
        feed = feedparser.parse(url_feed)
        urls = []
        for entry in feed.entries:
            link = getattr(entry, "link", None)
            if link:
                urls.append(link)
        return urls
    except Exception as e:
        print(f"  [ERROR] Gagal parse feed: {e}")
        return []


# =============================================================================
# FUNGSI UTAMA: SCRAPING
# =============================================================================

def jalankan_scraping():
    """
    Fungsi utama yang menjalankan seluruh proses scraping.
    Loop melalui setiap RSS feed sampai target artikel tercapai.
    """
    print("=" * 60)
    print("  SCRAPING BERITA INDONESIA - TOPIC MODELLING DATASET")
    print(f"  Target  : {TARGET_ARTIKEL} artikel")
    print(f"  Output  : {OUTPUT_FILE}")
    print(f"  Mulai   : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 60)

    # Kumpulkan data di sini
    data_artikel   = []   # List of string (isi artikel)
    hash_duplikat  = set() # Set hash untuk cek duplikat
    total_gagal    = 0    # Counter URL yang gagal
    total_dicoba   = 0    # Counter URL yang dicoba

    # Loop setiap RSS feed
    for nama_media, url_feed in RSS_FEEDS:

        # Cek apakah target sudah tercapai
        if len(data_artikel) >= TARGET_ARTIKEL:
            break

        print(f"\n[FEED] {nama_media}")
        print(f"  URL  : {url_feed}")

        # Ambil daftar URL dari feed
        daftar_url = ambil_url_dari_feed(url_feed)
        print(f"  Artikel ditemukan di feed: {len(daftar_url)}")

        if not daftar_url:
            print(f"  [SKIP] Feed kosong atau tidak bisa diakses.")
            time.sleep(JEDA_ANTAR_FEED)
            continue

        # Loop setiap URL artikel dalam feed
        for url_artikel in daftar_url:

            # Stop jika target tercapai
            if len(data_artikel) >= TARGET_ARTIKEL:
                break

            total_dicoba += 1
            nomor_sekarang = len(data_artikel) + 1

            # Tampilkan progress counter
            print(
                f"  [{nomor_sekarang:04d}/{TARGET_ARTIKEL}] "
                f"Mencoba: {url_artikel[:70]}...",
                end="", flush=True
            )

            # --- Ekstrak teks artikel ---
            teks = ekstrak_artikel(url_artikel)

            # Validasi 1: Teks tidak kosong
            if not teks:
                print(" -> [GAGAL] Teks kosong")
                total_gagal += 1
                time.sleep(JEDA_ANTAR_URL)
                continue

            # Bersihkan teks
            teks = bersihkan_teks(teks)

            # Validasi 2: Minimal panjang karakter
            if len(teks) < MIN_KARAKTER:
                print(f" -> [SKIP] Terlalu pendek ({len(teks)} kar)")
                total_gagal += 1
                time.sleep(JEDA_ANTAR_URL)
                continue

            # Validasi 3: Cek duplikat via hash
            h = hash_teks(teks)
            if h in hash_duplikat:
                print(" -> [SKIP] Duplikat")
                time.sleep(JEDA_ANTAR_URL)
                continue

            # Validasi 4: Cek bahasa Indonesia
            if not adalah_bahasa_indonesia(teks):
                print(" -> [SKIP] Bukan Bahasa Indonesia")
                total_gagal += 1
                time.sleep(JEDA_ANTAR_URL)
                continue

            # Semua validasi lolos — simpan artikel
            hash_duplikat.add(h)
            data_artikel.append(teks)
            print(f" -> [OK] {len(teks)} karakter (total: {len(data_artikel)})")

            # Jeda antar request (rate limiting)
            time.sleep(JEDA_ANTAR_URL)

        # Jeda lebih lama saat pindah ke feed berikutnya
        time.sleep(JEDA_ANTAR_FEED)

    # ==========================================================================
    # SIMPAN KE CSV
    # ==========================================================================
    print("\n" + "=" * 60)
    print("  MENYIMPAN HASIL...")
    print("=" * 60)

    if data_artikel:
        df = pd.DataFrame({"isi": data_artikel})
        df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
        # utf-8-sig menambahkan BOM agar Excel bisa baca karakter Indonesia dengan benar
        print(f"  File disimpan: {OUTPUT_FILE}")
    else:
        print("  [WARN] Tidak ada artikel yang berhasil dikumpulkan!")

    # ==========================================================================
    # RINGKASAN AKHIR
    # ==========================================================================
    print("\n" + "=" * 60)
    print("  RINGKASAN AKHIR")
    print("=" * 60)
    print(f"  Total URL dicoba          : {total_dicoba}")
    print(f"  Total artikel BERHASIL    : {len(data_artikel)}")
    print(f"  Total artikel GAGAL/SKIP  : {total_gagal}")
    print(f"  Target tercapai           : {'YA ✓' if len(data_artikel) >= TARGET_ARTIKEL else 'BELUM ✗'}")
    if data_artikel:
        print(f"  Rata-rata panjang teks    : {sum(len(t) for t in data_artikel) // len(data_artikel)} karakter")
    print(f"  Selesai                   : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 60)

    if len(data_artikel) < TARGET_ARTIKEL:
        kekurangan = TARGET_ARTIKEL - len(data_artikel)
        print(f"\n[INFO] Kurang {kekurangan} artikel dari target.")
        print("  Saran:")
        print("  1. Tambah RSS feed lain di daftar RSS_FEEDS")
        print("  2. Kurangi MIN_KARAKTER (misal jadi 100)")
        print("  3. Jalankan ulang script — feed yang sama bisa punya artikel berbeda")

    return data_artikel


# =============================================================================
# ENTRY POINT
# =============================================================================
if __name__ == "__main__":
    # Cek library penting
    if not NEWSPAPER_OK:
        print("[ERROR] newspaper4k diperlukan. Install dulu:")
        print("  pip install newspaper4k")
        sys.exit(1)

    jalankan_scraping()

  SCRAPING BERITA INDONESIA - TOPIC MODELLING DATASET
  Target  : 1000 artikel
  Output  : dataset_berita_2.csv
  Mulai   : 2026-04-24 23:28:26

[FEED] Antara - Terkini
  URL  : https://www.antaranews.com/rss/terkini.xml
  Artikel ditemukan di feed: 50
  [0001/1000] Mencoba: https://www.antaranews.com/video/5541679/picu-ketegangan-dengan-korut-... -> [OK] 352 karakter (total: 1)
  [0002/1000] Mencoba: https://www.antaranews.com/berita/5541676/maruarar-tekankan-pemberdaya... -> [OK] 3476 karakter (total: 2)
  [0003/1000] Mencoba: https://www.antaranews.com/video/5541664/swimside-hadirkan-pengalaman-... -> [OK] 396 karakter (total: 3)
  [0004/1000] Mencoba: https://www.antaranews.com/berita/5541675/alwi-farhan-dan-raymond-joaq... -> [OK] 2242 karakter (total: 4)
  [0005/1000] Mencoba: https://www.antaranews.com/berita/5541673/esdm-tawarkan-116-blok-migas... -> [OK] 3404 karakter (total: 5)
  [0006/1000] Mencoba: https://www.antaranews.com/berita/5541669/bpom-perkuat-regulasi-kolabo... ->